In [1]:
import numpy as np
import itertools

In [2]:
# Evidence 1 (strong)
LR1_pos = 0.95 / 0.05   # incriminating
LR1_neg = 0.05 / 0.95   # exculpatory

# Evidence 2 (weak)
LR2_pos = 0.51 / 0.49
LR2_neg = 0.49 / 0.51

# Conviction threshold
threshold = 0.95

In [3]:
def posterior_prob(odds):
    return odds / (1 + odds)

In [4]:
# List of possible outcomes for E1 and E2
outcomes = list(itertools.product(['+', '-'], repeat=2))
print(outcomes)
# [('+' , '+'), ('+' , '-'), ('-' , '+'), ('-' , '-')]

[('+', '+'), ('+', '-'), ('-', '+'), ('-', '-')]


In [5]:
results = []

for e1, e2 in outcomes:
    # Compute odds
    odds = 1  # prior odds = 1:1
    if e1 == '+':
        odds *= LR1_pos
    else:
        odds *= LR1_neg
    
    if e2 == '+':
        odds *= LR2_pos
    else:
        odds *= LR2_neg
    
    # Posterior probability
    p_guilty = posterior_prob(odds)
    
    # Decision: convict if above threshold
    decision = 'Convict' if p_guilty >= threshold else 'Acquit'
    
    results.append({
        'E1': e1,
        'E2': e2,
        'odds': odds,
        'P(G|E1,E2)': p_guilty,
        'Decision': decision
    })

for r in results:
    print(r)


{'E1': '+', 'E2': '+', 'odds': 19.77551020408163, 'P(G|E1,E2)': 0.9518664047151277, 'Decision': 'Convict'}
{'E1': '+', 'E2': '-', 'odds': 18.25490196078431, 'P(G|E1,E2)': 0.9480651731160896, 'Decision': 'Acquit'}
{'E1': '-', 'E2': '+', 'odds': 0.054779806659505915, 'P(G|E1,E2)': 0.05193482688391039, 'Decision': 'Acquit'}
{'E1': '-', 'E2': '-', 'odds': 0.05056759545923632, 'P(G|E1,E2)': 0.048133595284872294, 'Decision': 'Acquit'}


In [6]:
P_G = 0.5  # guilty
P_I = 1 - P_G  # innocent

In [7]:
# Sensitivity & specificity for each outcome
accuracy_results = []

for r in results:
    # Sensitivity = P(convict | guilty)
    if r['Decision'] == 'Convict':
        sens = r['P(G|E1,E2)']  # probability guilty given evidence
        spec = 1 - r['P(G|E1,E2)']  # probability innocent given evidence → acquit
    else:
        sens = 1 - r['P(G|E1,E2)']
        spec = r['P(G|E1,E2)']
    
    accuracy_results.append({
        'E1': r['E1'],
        'E2': r['E2'],
        'Decision': r['Decision'],
        'P(G|E1,E2)': r['P(G|E1,E2)'],
        'Accuracy_given_guilty': sens,
        'Accuracy_given_innocent': spec
    })

for a in accuracy_results:
    print(a)


{'E1': '+', 'E2': '+', 'Decision': 'Convict', 'P(G|E1,E2)': 0.9518664047151277, 'Accuracy_given_guilty': 0.9518664047151277, 'Accuracy_given_innocent': 0.048133595284872266}
{'E1': '+', 'E2': '-', 'Decision': 'Acquit', 'P(G|E1,E2)': 0.9480651731160896, 'Accuracy_given_guilty': 0.05193482688391038, 'Accuracy_given_innocent': 0.9480651731160896}
{'E1': '-', 'E2': '+', 'Decision': 'Acquit', 'P(G|E1,E2)': 0.05193482688391039, 'Accuracy_given_guilty': 0.9480651731160896, 'Accuracy_given_innocent': 0.05193482688391039}
{'E1': '-', 'E2': '-', 'Decision': 'Acquit', 'P(G|E1,E2)': 0.048133595284872294, 'Accuracy_given_guilty': 0.9518664047151277, 'Accuracy_given_innocent': 0.048133595284872294}


In [8]:
expected_accuracy = 0
for r in accuracy_results:
    # probability of this evidence combination
    # P(E1,E2 | G) and P(E1,E2 | I)
    if r['E1'] == '+':
        p_e1_given_g = 0.95
        p_e1_given_i = 0.05
    else:
        p_e1_given_g = 0.05
        p_e1_given_i = 0.95
    
    if r['E2'] == '+':
        p_e2_given_g = 0.51
        p_e2_given_i = 0.49
    else:
        p_e2_given_g = 0.49
        p_e2_given_i = 0.51
    
    p_comb_given_g = p_e1_given_g * p_e2_given_g
    p_comb_given_i = p_e1_given_i * p_e2_given_i
    
    # Weighted by prior
    expected_accuracy += P_G * p_comb_given_g * r['Accuracy_given_guilty']
    expected_accuracy += P_I * p_comb_given_i * r['Accuracy_given_innocent']

print("Expected accuracy with E1 and E2:", expected_accuracy)


Expected accuracy with E1 and E2: 0.3028513238289206


In [19]:
# Expected accuracy with only E1
expected_acc_E1 = P_G * (0.95) + P_I * (0.95)  # sensitivity & specificity
print("Expected accuracy with only E1:", expected_acc_E1)


Expected accuracy with only E1: 0.95
